# NB23 — S9 saved-mask geometry baseline

**CPU / Accelerator None · ONE copy · Internet ON · enable HF_TOKEN · Run All.**
**Attach nothing.** No dataset, weights, GPU or new training is needed.

Uses the verified NB22 annotation revision `4110bdcc…` and existing SegFormer-B0
seed-1 predictions, pinned to HF commit `05bf37c0…`. It does not silently select
new labels. P06 is excluded from main scores while review is unresolved; its
mask/points remain visible as a diagnostic. You can run this now with 11/12 labels.

Checks native image identity and per-run validation membership. Existing fold
0/2 leakage caveats still apply: this is not independent new-tyre accuracy.
Extracts left/right mask extrema at the same three guide rows. Exact frame-edge
and empty-row predictions are rejected and reduce coverage. No threshold tuning.

Results: SUMMARY.json, all 72 POINTS.json records with exclusion reasons,
12 mask/point PNGs (cyan human, yellow predicted), source hashes and contract.
These are predicted-mask diagrams, not original-image overlays or manual masks.
Read error together with coverage. No automated conclusion that HRNet is useful
or unnecessary; PatchCore and full S9 remain uncompleted.

Small JSON sources are cached; rerunning deterministically rebuilds the same
content-addressed output, without repeating training or downloads in-session.
A fresh session refetches only small saved predictions. Uploads one batch at
completion, every 30 minutes if needed, and on catchable Stop/error. Forced kernel
kills cannot flush. Retry Run All after a failed upload. Do not run four workers.
Source and output caps: 20 MiB each, comfortably below Kaggle working storage.
Dependencies are isolated; existing Transformers/HF packages are not downgraded.


In [1]:
import sys, subprocess, base64, importlib
from pathlib import Path
WORK=Path('/kaggle/working/s9_geometry'); WORK.mkdir(exist_ok=True)
# Run in a child with isolated dependency precedence; preserve Kaggle's base env.
DEPS=WORK/'deps'
if not (DEPS/'pycocotools').exists() or not (DEPS/'huggingface_hub').exists():
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--target',str(DEPS),
        'numpy<3','Pillow>=10,<13','requests>=2.32,<3','pycocotools==2.0.11','huggingface_hub>=1.3,<2'])
(WORK/'s9_pilot.py').write_bytes(base64.b64decode('IiIiU21hbGwsIGxvc3NsZXNzIFM5IGFubm90YXRpb24gZmVhc2liaWxpdHkgcGlsb3QuIE5vIG1vZGVsIHRyYWluaW5nIG9yIGhlYWx0aCBpbmZlcmVuY2UuIiIiCmltcG9ydCBiYXNlNjQKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNodXRpbAppbXBvcnQgdGltZQppbXBvcnQgemlwZmlsZQpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKClZFUlNJT04gPSAnczktaW5wdXQtcGlsb3QtcjEnClJFUE8gPSAnU2hhbm11azQ2MjIvdHlyZS13ZWFyLXN0dWR5JwpNQVhfQllURVMgPSAyMCAqIDEwMjQgKiAxMDI0ClBPSU5UUyA9IFsnbGVmdF91cHBlcicsICdyaWdodF91cHBlcicsICdsZWZ0X21pZGRsZScsICdyaWdodF9taWRkbGUnLCAnbGVmdF9sb3dlcicsICdyaWdodF9sb3dlciddClNUQVRFUyA9IFsndmlzaWJsZScsICdvY2NsdWRlZCcsICdvdXRzaWRlX2ZyYW1lJywgJ3VuY2VydGFpbiddClRSSUFHRSA9IFsnbm9fdmlzaWJsZV9pc3N1ZScsICd2aXNpYmxlX2lzc3VlJywgJ3VuYXNzZXNzYWJsZSddCgpkZWYgZGlnZXN0KHJhdyk6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYocmF3KS5oZXhkaWdlc3QoKQoKZGVmIGNhbm9uaWNhbCh2YWx1ZSk6CiAgICByZXR1cm4ganNvbi5kdW1wcyh2YWx1ZSwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCcsJywgJzonKSwgZW5zdXJlX2FzY2lpPUZhbHNlKS5lbmNvZGUoKQoKZGVmIHdyaXRlX2pzb24ocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCk7IHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRtcCA9IHBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXggKyAnLnRtcCcpCiAgICB0bXAud3JpdGVfYnl0ZXMoY2Fub25pY2FsKHZhbHVlKSk7IHRtcC5yZXBsYWNlKHBhdGgpCgpkZWYgZGlzY292ZXIocm9vdD0nL2thZ2dsZS9pbnB1dCcpOgogICAgY2FuZGlkYXRlcyA9IHNvcnRlZChQYXRoKHJvb3QpLmdsb2IoJyoqL21hbmlmZXN0cy9jbGVhbl9tYW5pZmVzdC5jc3YnKSkKICAgIGlmIGxlbihjYW5kaWRhdGVzKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0F0dGFjaCBPTkUgVGlyZSBEYXRhc2V0IFByZXBhcmVkIHBhY2thZ2UsIG9yIHNldCBEQVRBX1JPT1QgdG8gaXRzIEZJTkFMIGRpcmVjdG9yeS4nKQogICAgcmV0dXJuIGNhbmRpZGF0ZXNbMF0ucGFyZW50LnBhcmVudAoKZGVmIHByZXBhcmUoZGF0YV9yb290LCBvdXRwdXQsIHRlbXBsYXRlKToKICAgIGRhdGFfcm9vdCwgb3V0cHV0ID0gUGF0aChkYXRhX3Jvb3QpLCBQYXRoKG91dHB1dCkKICAgIHdpdGggKGRhdGFfcm9vdC8nbWFuaWZlc3RzL2NsZWFuX21hbmlmZXN0LmNzdicpLm9wZW4oZW5jb2Rpbmc9J3V0Zi04LXNpZycsIG5ld2xpbmU9JycpIGFzIGY6CiAgICAgICAgYWxsX3Jvd3MgPSBsaXN0KGNzdi5EaWN0UmVhZGVyKGYpKQogICAgaWYgbGVuKGFsbF9yb3dzKSAhPSA0MTggb3IgbGVuKHtyWydpbWFnZV9pZCddIGZvciByIGluIGFsbF9yb3dzfSkgIT0gNDE4OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0V4cGVjdGVkIHRoZSBmcm96ZW4gNDE4IHVuaXF1ZSBjbGVhbiBvcmlnaW5hbHMuJykKICAgIGdyb3VwcyA9IHNvcnRlZCh7clsnc2Vzc2lvbl9ncm91cCddIGZvciByIGluIGFsbF9yb3dzfSkKICAgIGlmIGxlbihncm91cHMpICE9IDEyOiByYWlzZSBWYWx1ZUVycm9yKCdFeHBlY3RlZCAxMiBjYXB0dXJlIHNlc3Npb25zLCBub3QgYSBuZXcgZGF0YXNldC4nKQogICAgY2hvc2VuID0gW10KICAgIGZvciBncm91cCBpbiBncm91cHM6CiAgICAgICAgcnIgPSBzb3J0ZWQoW3IgZm9yIHIgaW4gYWxsX3Jvd3MgaWYgclsnc2Vzc2lvbl9ncm91cCddID09IGdyb3VwXSwga2V5PWxhbWJkYSByOiByWydpbWFnZV9pZCddKQogICAgICAgIGNob3Nlbi5hcHBlbmQocnJbbGVuKHJyKS8vMl0pICAjIEZpeGVkIG1lZGlhbi1JRCByZXByZXNlbnRhdGl2ZSwgbm90IHNlbGVjdGVkIGJ5IG1vZGVsIHBlcmZvcm1hbmNlLgogICAgcmVjb3JkcywgZW1iZWRkZWQgPSBbXSwgW10KICAgIGZvciBpLCByIGluIGVudW1lcmF0ZShjaG9zZW4pOgogICAgICAgIHAgPSAoZGF0YV9yb290L3JbJ3JlbGF0aXZlX3BhdGgnXSkucmVzb2x2ZSgpCiAgICAgICAgaWYgbm90IHAuaXNfcmVsYXRpdmVfdG8oZGF0YV9yb290LnJlc29sdmUoKSk6IHJhaXNlIFZhbHVlRXJyb3IoJ0ltYWdlIHBhdGggZXNjYXBlcyBkYXRhIHJvb3QnKQogICAgICAgIHJhdyA9IHAucmVhZF9ieXRlcygpCiAgICAgICAgaWYgZGlnZXN0KHJhdykgIT0gclsnZmlsZV9zaGEyNTYnXTogcmFpc2UgVmFsdWVFcnJvcihmJ0ltYWdlIGhhc2ggbWlzbWF0Y2g6IHBpbG90IHtpKzF9JykKICAgICAgICB3aXRoIEltYWdlLm9wZW4ocCkgYXMgaW06CiAgICAgICAgICAgIGlmIGltLmZvcm1hdCAhPSAnSlBFRycgb3IgaW0uc2l6ZSAhPSAoaW50KHJbJ3dpZHRoJ10pLCBpbnQoclsnaGVpZ2h0J10pKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0V4cGVjdGVkIHVuY2hhbmdlZCBuYXRpdmUgSlBFRyBkaW1lbnNpb25zJykKICAgICAgICAgICAgd2lkdGgsIGhlaWdodCA9IGltLnNpemUKICAgICAgICBpdGVtID0gZGljdChwaWxvdF9pZD1mJ1B7aSsxOjAyZH0nLCBpbWFnZV9pZD1yWydpbWFnZV9pZCddLCBzZXNzaW9uPXJbJ3Nlc3Npb25fZ3JvdXAnXSwKICAgICAgICAgICAgICAgICAgICBmb2xkPWludChyWydmb2xkX2lkJ10pLCBvcmlnaW5hbF9yZWxhdGl2ZV9wYXRoPXJbJ3JlbGF0aXZlX3BhdGgnXSwKICAgICAgICAgICAgICAgICAgICBpbWFnZV9zaGEyNTY9ZGlnZXN0KHJhdyksIHdpZHRoPXdpZHRoLCBoZWlnaHQ9aGVpZ2h0LAogICAgICAgICAgICAgICAgICAgIGd1aWRlX3k9W3JvdW5kKChoZWlnaHQtMSkqZikgZm9yIGYgaW4gKC4yNSwuNSwuNzUpXSwKICAgICAgICAgICAgICAgICAgICB0cmFuc2Zvcm09J2lkZW50aXR5OyBuYXRpdmUgcGl4ZWxzOyBubyBjcm9wLCByZXNpemUgb3IgcmVjb21wcmVzc2lvbicpCiAgICAgICAgcmVjb3Jkcy5hcHBlbmQoaXRlbSkKICAgICAgICBlbWJlZGRlZC5hcHBlbmQoe2s6aXRlbVtrXSBmb3IgayBpbiBbJ3BpbG90X2lkJywnaW1hZ2Vfc2hhMjU2Jywnd2lkdGgnLCdoZWlnaHQnLCdndWlkZV95J119CiAgICAgICAgICAgICAgICAgICAgICAgIHwgeydpbWFnZSc6J2RhdGE6aW1hZ2UvanBlZztiYXNlNjQsJytiYXNlNjQuYjY0ZW5jb2RlKHJhdykuZGVjb2RlKCl9KQogICAgdGVtcGxhdGUgPSBQYXRoKHRlbXBsYXRlKS5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04JykKICAgIHByb3RvY29sID0gZGljdCh2ZXJzaW9uPVZFUlNJT04sIG1hbmlmZXN0X3NoYTI1Nj1kaWdlc3QoKGRhdGFfcm9vdC8nbWFuaWZlc3RzL2NsZWFuX21hbmlmZXN0LmNzdicpLnJlYWRfYnl0ZXMoKSksCiAgICAgICAgICAgICAgICAgICAgc291cmNlX3NoYTI1Nj1kaWdlc3QoUGF0aChfX2ZpbGVfXykucmVhZF9ieXRlcygpKSwgdGVtcGxhdGVfc2hhMjU2PWRpZ2VzdCh0ZW1wbGF0ZS5lbmNvZGUoKSksCiAgICAgICAgICAgICAgICAgICAgcG9pbnRzPVBPSU5UUywgaW1hZ2VzPXJlY29yZHMsIGNvdW50PTEyLAogICAgICAgICAgICAgICAgICAgIHB1cnBvc2U9J0ZlYXNpYmlsaXR5IG9ubHk7IHByb3Bvc2VkIGltYWdlLXBsYW5lIHRyZWFkLWJvdW5kYXJ5IGxhYmVsczsgbm8gdHJhaW5pbmcgYXBwcm92YWwnLAogICAgICAgICAgICAgICAgICAgIGhlYWx0aHlfcmVmZXJlbmNlX2VsaWdpYmxlPUZhbHNlLCBodW1hbl9yZXZpZXdfcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhY2thZ2VfaWQgPSBkaWdlc3QoY2Fub25pY2FsKHByb3RvY29sKSkKICAgIHByb3RvY29sWydwYWNrYWdlX2lkJ10gPSBwYWNrYWdlX2lkCiAgICBkZXN0ID0gb3V0cHV0L3BhY2thZ2VfaWQ7IGRlc3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcGF5bG9hZCA9IGRpY3QocGFja2FnZV9pZD1wYWNrYWdlX2lkLCB2ZXJzaW9uPVZFUlNJT04sIHBvaW50cz1QT0lOVFMsIGltYWdlcz1lbWJlZGRlZCkKICAgIHBhZ2UgPSB0ZW1wbGF0ZS5yZXBsYWNlKCdfX1BJTE9UX0RBVEFfXycsIGpzb24uZHVtcHMocGF5bG9hZCkucmVwbGFjZSgnPCcsJ1xcdTAwM2MnKSkKICAgIGlmIGxlbihwYWdlLmVuY29kZSgpKSA+IE1BWF9CWVRFUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdMb3NzbGVzcyAxMi1pbWFnZSBwYWdlIGV4Y2VlZHMgMjAgTWlCOiBzdG9wcGVkIHdpdGhvdXQgcmVkdWNpbmcgZGV0YWlsLiBSZXF1ZXN0IGEgc21hbGxlciBiYXRjaC4nKQogICAgKGRlc3QvJ0FOTk9UQVRFLmh0bWwnKS53cml0ZV90ZXh0KHBhZ2UsIGVuY29kaW5nPSd1dGYtOCcpCiAgICB3cml0ZV9qc29uKGRlc3QvJ01BTklGRVNULmpzb24nLCBwcm90b2NvbCkKICAgIHppcF9wYXRoID0gZGVzdC8nUElMT1RfMTJfSU1BR0VTLnppcCcKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKHppcF9wYXRoLCAndycsIGNvbXByZXNzaW9uPXppcGZpbGUuWklQX0RFRkxBVEVEKSBhcyBhcmNoaXZlOgogICAgICAgIGZvciBuYW1lIGluIFsnQU5OT1RBVEUuaHRtbCcsJ01BTklGRVNULmpzb24nXToKICAgICAgICAgICAgaW5mbz16aXBmaWxlLlppcEluZm8obmFtZSxkYXRlX3RpbWU9KDIwMjYsOSwxNCwwLDAsMCkpCiAgICAgICAgICAgIGluZm8uY29tcHJlc3NfdHlwZT16aXBmaWxlLlpJUF9ERUZMQVRFRAogICAgICAgICAgICBhcmNoaXZlLndyaXRlc3RyKGluZm8sKGRlc3QvbmFtZSkucmVhZF9ieXRlcygpKQogICAgaWYgemlwX3BhdGguc3RhdCgpLnN0X3NpemUgPiBNQVhfQllURVM6IHJhaXNlIFZhbHVlRXJyb3IoJ1BhY2thZ2UgZXhjZWVkcyAyMCBNaUI7IG5vIHVwbG9hZCBhbGxvd2VkJykKICAgIHdyaXRlX2pzb24oZGVzdC8nU1RBVFVTLmpzb24nLCBkaWN0KHZlcnNpb249VkVSU0lPTiwgcGFja2FnZV9pZD1wYWNrYWdlX2lkLAogICAgICAgIHN0YXR1cz0nYXdhaXRpbmdfcGlsb3RfYW5ub3RhdGlvbicsIGltYWdlX2NvdW50PTEyLCB6aXBfYnl0ZXM9emlwX3BhdGguc3RhdCgpLnN0X3NpemUsCiAgICAgICAgaHRtbF9ieXRlcz0oZGVzdC8nQU5OT1RBVEUuaHRtbCcpLnN0YXQoKS5zdF9zaXplLCB6aXBfc2hhMjU2PWRpZ2VzdCh6aXBfcGF0aC5yZWFkX2J5dGVzKCkpLAogICAgICAgIGZ1bGxfczlfY29tcGxldGU9RmFsc2UsIHRyYWluaW5nX2FwcHJvdmVkPUZhbHNlLCBoZWFsdGh5X3JlZmVyZW5jZV9lbGlnaWJsZT1GYWxzZSkpCiAgICBwcmludChmJzEyIG5hdGl2ZSBpbWFnZXM7IFpJUCB7emlwX3BhdGguc3RhdCgpLnN0X3NpemUvMioqMjA6LjJmfSBNaUIuIE9wZW4gQU5OT1RBVEUuaHRtbCBhZnRlciBleHRyYWN0aW9uLicpCiAgICByZXR1cm4gZGVzdAoKZGVmIHZhbGlkYXRlX2Fubm90YXRpb25zKHZhbHVlLCBtYW5pZmVzdCk6CiAgICBpZiBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgZGljdCkgb3IgdmFsdWUuZ2V0KCdwYWNrYWdlX2lkJykgIT0gbWFuaWZlc3RbJ3BhY2thZ2VfaWQnXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdBbm5vdGF0aW9uIHBhY2thZ2UgaWRlbnRpdHkgbWlzbWF0Y2guIEltcG9ydCB0aGUgSlNPTiBpbnRvIHRoZSBtYXRjaGluZyBwYWdlLicpCiAgICBpZiB2YWx1ZS5nZXQoJ3ZlcnNpb24nKSAhPSBWRVJTSU9OOiByYWlzZSBWYWx1ZUVycm9yKCdBbm5vdGF0aW9uIHNjaGVtYSBtaXNtYXRjaCcpCiAgICBsYWJlbHMgPSB2YWx1ZS5nZXQoJ2Fubm90YXRpb25zJykKICAgIGlmIG5vdCBpc2luc3RhbmNlKGxhYmVscywgbGlzdCkgb3IgbGVuKGxhYmVscykgPiAxMjogcmFpc2UgVmFsdWVFcnJvcignRXhwZWN0ZWQgYXQgbW9zdCAxMiBhbm5vdGF0aW9uIHJlY29yZHMnKQogICAgYnlfaWQgPSB7clsncGlsb3RfaWQnXTpyIGZvciByIGluIG1hbmlmZXN0WydpbWFnZXMnXX07IHNlZW49c2V0KCk7IGNoZWNrZWQ9W10KICAgIGZvciBhIGluIGxhYmVsczoKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShhLGRpY3QpOiByYWlzZSBWYWx1ZUVycm9yKCdJbnZhbGlkIGFubm90YXRpb24gZW50cnknKQogICAgICAgIHBpZD1hLmdldCgncGlsb3RfaWQnKQogICAgICAgIGlmIHBpZCBub3QgaW4gYnlfaWQgb3IgcGlkIGluIHNlZW46IHJhaXNlIFZhbHVlRXJyb3IoJ1Vua25vd24gb3IgZHVwbGljYXRlZCBwaWxvdCBJRCcpCiAgICAgICAgc2Vlbi5hZGQocGlkKTsgcmVmPWJ5X2lkW3BpZF0KICAgICAgICBpZiBhLmdldCgnaW1hZ2Vfc2hhMjU2JykgIT0gcmVmWydpbWFnZV9zaGEyNTYnXTogcmFpc2UgVmFsdWVFcnJvcignSW1hZ2UgaWRlbnRpdHkgbWlzbWF0Y2gnKQogICAgICAgIHBvaW50cz1hLmdldCgncG9pbnRzJyx7fSkKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShwb2ludHMsZGljdCkgb3Igc2V0KHBvaW50cyktc2V0KFBPSU5UUyk6IHJhaXNlIFZhbHVlRXJyb3IoJ1Vua25vd24gbGFuZG1hcmsgbmFtZXMnKQogICAgICAgIGNvbXBsZXRlPVRydWU7IHZpc2libGU9MAogICAgICAgIGZvciBqLG5hbWUgaW4gZW51bWVyYXRlKFBPSU5UUyk6CiAgICAgICAgICAgIHA9cG9pbnRzLmdldChuYW1lKQogICAgICAgICAgICBpZiBwIGlzIE5vbmU6IGNvbXBsZXRlPUZhbHNlOyBjb250aW51ZQogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShwLGRpY3QpIG9yIHAuZ2V0KCdzdGF0ZScpIG5vdCBpbiBTVEFURVM6IHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgdmlzaWJpbGl0eSBzdGF0ZScpCiAgICAgICAgICAgIGlmIHBbJ3N0YXRlJ109PSd2aXNpYmxlJzoKICAgICAgICAgICAgICAgIHgseT1wLmdldCgneCcpLHAuZ2V0KCd5JykKICAgICAgICAgICAgICAgIGlmIGFueSh0eXBlKHYpIG5vdCBpbiAoZmxvYXQsaW50KSBvciBub3QgbWF0aC5pc2Zpbml0ZSh2KSBmb3IgdiBpbiAoeCx5KSk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVmlzaWJsZSBwb2ludHMgbmVlZCBmaW5pdGUgY29vcmRpbmF0ZXMnKQogICAgICAgICAgICAgICAgaWYgbm90ICgwPD14PHJlZlsnd2lkdGgnXSBhbmQgMDw9eTxyZWZbJ2hlaWdodCddKTogcmFpc2UgVmFsdWVFcnJvcignUG9pbnQgb3V0IG9mIGJvdW5kcycpCiAgICAgICAgICAgICAgICBpZiB5ICE9IHJlZlsnZ3VpZGVfeSddW2ovLzJdOiByYWlzZSBWYWx1ZUVycm9yKCdQb2ludCBtdXN0IGJlIG9uIGl0cyBleGFjdCBob3Jpem9udGFsIGd1aWRlJykKICAgICAgICAgICAgICAgIHZpc2libGUrPTEKICAgICAgICAgICAgZWxpZiBwLmdldCgneCcpIGlzIG5vdCBOb25lIG9yIHAuZ2V0KCd5JykgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbnZpc2libGUgcG9pbnQgbXVzdCBub3QgaGF2ZSBndWVzc2VkIGNvb3JkaW5hdGVzJykKICAgICAgICBmb3IgbGV2ZWwgaW4gWyd1cHBlcicsJ21pZGRsZScsJ2xvd2VyJ106CiAgICAgICAgICAgIGwscj1wb2ludHMuZ2V0KCdsZWZ0XycrbGV2ZWwse30pLHBvaW50cy5nZXQoJ3JpZ2h0XycrbGV2ZWwse30pCiAgICAgICAgICAgIGlmIGwuZ2V0KCdzdGF0ZScpPT1yLmdldCgnc3RhdGUnKT09J3Zpc2libGUnIGFuZCBsWyd4J10+PXJbJ3gnXToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0xlZnQvcmlnaHQgcG9pbnRzIGFyZSBjcm9zc2VkIG9yIGVxdWFsJykKICAgICAgICB0cmlhZ2U9YS5nZXQoJ3Zpc3VhbF9yZXZpZXcnKQogICAgICAgIGlmIHRyaWFnZSBpcyBub3QgTm9uZSBhbmQgdHJpYWdlIG5vdCBpbiBUUklBR0U6IHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgdmlzdWFsIHJldmlldycpCiAgICAgICAgaWYgdHJpYWdlIGlzIE5vbmU6IGNvbXBsZXRlPUZhbHNlCiAgICAgICAgbm90ZT1hLmdldCgnbm90ZScsJycpCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2Uobm90ZSxzdHIpIG9yIGxlbihub3RlKT4yMDAwOiByYWlzZSBWYWx1ZUVycm9yKCdOb3RlIGV4Y2VlZHMgMjAwMCBjaGFyYWN0ZXJzJykKICAgICAgICBpZiB0cmlhZ2U9PSd2aXNpYmxlX2lzc3VlJyBhbmQgbm90IG5vdGUuc3RyaXAoKTogY29tcGxldGU9RmFsc2UKICAgICAgICBldmlkZW5jZT1hLmdldCgnaW5kZXBlbmRlbnRfcmVjb3JkJywndW5rbm93bicpCiAgICAgICAgaWYgZXZpZGVuY2Ugbm90IGluIFsndW5rbm93bicsJ2F2YWlsYWJsZSddOiByYWlzZSBWYWx1ZUVycm9yKCdJbnZhbGlkIGV2aWRlbmNlIGF2YWlsYWJpbGl0eScpCiAgICAgICAgIyBBIHVzZXIgYXNzZXJ0aW9uIGlzIHJlY29yZGVkLCBORVZFUiBhdXRvbWF0aWNhbGx5IHByb21vdGVkIHRvIHZlcmlmaWVkIGhlYWx0aC4KICAgICAgICBjaGVja2VkLmFwcGVuZChkaWN0KHBpbG90X2lkPXBpZCwgaW1hZ2VfaWQ9cmVmWydpbWFnZV9pZCddLCBjb21wbGV0ZT1jb21wbGV0ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHZpc2libGVfcG9pbnRzPXZpc2libGUsIHZpc3VhbF9yZXZpZXc9dHJpYWdlLCBoZWFsdGh5X3JlZmVyZW5jZV9lbGlnaWJsZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGluZGVwZW5kZW50X3JlY29yZD1ldmlkZW5jZSwgbm90ZT1ub3RlKSkKICAgIHJldHVybiBjaGVja2VkCgpkZWYgcmV2aWV3KGFubm90YXRpb25fcGF0aCwgcGFja2FnZSwgb3V0cHV0KToKICAgIHBhY2thZ2UsIG91dHB1dD1QYXRoKHBhY2thZ2UpLFBhdGgob3V0cHV0KQogICAgcmF3PVBhdGgoYW5ub3RhdGlvbl9wYXRoKS5yZWFkX2J5dGVzKCkKICAgIGlmIGxlbihyYXcpPjEwMjQqMTAyNDogcmFpc2UgVmFsdWVFcnJvcignVXBsb2FkIHRoZSBhbm5vdGF0aW9uLW9ubHkgSlNPTiAodW5kZXIgMSBNaUIpLCBub3QgaW1hZ2VzIG9yIFpJUCcpCiAgICB2YWx1ZT1qc29uLmxvYWRzKHJhdyk7IG1hbmlmZXN0PWpzb24ubG9hZHMoKHBhY2thZ2UvJ01BTklGRVNULmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGlmIGRpZ2VzdChjYW5vbmljYWwoe2s6diBmb3Igayx2IGluIG1hbmlmZXN0Lml0ZW1zKCkgaWYgayE9J3BhY2thZ2VfaWQnfSkpIT1tYW5pZmVzdC5nZXQoJ3BhY2thZ2VfaWQnKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdNYW5pZmVzdCBjb250ZW50IGhhc2ggbWlzbWF0Y2gnKQogICAgcmVzdWx0PXZhbGlkYXRlX2Fubm90YXRpb25zKHZhbHVlLG1hbmlmZXN0KQogICAgZGVzdD1vdXRwdXQvbWFuaWZlc3RbJ3BhY2thZ2VfaWQnXS8ncmV2aWV3cycvZGlnZXN0KHJhdyk7IGRlc3QubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICAoZGVzdC8nQU5OT1RBVElPTlMuanNvbicpLndyaXRlX2J5dGVzKHJhdykKICAgIGRvbmU9c3VtKHJbJ2NvbXBsZXRlJ10gZm9yIHIgaW4gcmVzdWx0KQogICAgd3JpdGVfanNvbihkZXN0LydSRVZJRVcuanNvbicsZGljdChzdGF0dXM9J25lZWRzX2h1bWFuX3JldmlldycgaWYgZG9uZT09MTIgZWxzZSAncGFydGlhbF9hbm5vdGF0aW9uJywKICAgICAgIGNvbXBsZXRlX2ltYWdlcz1kb25lLCBleHBlY3RlZF9pbWFnZXM9MTIsIHJlY29yZHM9cmVzdWx0LCBwYWNrYWdlX2lkPW1hbmlmZXN0WydwYWNrYWdlX2lkJ10sCiAgICAgICBhbm5vdGF0aW9uc19zaGEyNTY9ZGlnZXN0KHJhdyksIHRyYWluaW5nX2FwcHJvdmVkPUZhbHNlLCBoZWFsdGh5X3JlZmVyZW5jZV9lbGlnaWJsZT1GYWxzZSwKICAgICAgIG5leHRfYWN0aW9uPSdTZW5kIHRoZSBKU09OIGFuZCByZXZpZXcgcmVwb3J0IHRvIHRoZSBhc3Npc3RhbnQuIERvIG5vdCBsYWJlbCBtb3JlIG9yIHRyYWluIHlldC4nKSkKICAgIHByaW50KGYnUGlsb3QgY29tcGxldGlvbjoge2RvbmV9LzEyLiBNZWNoYW5pY2FsIGNoZWNrcyBvbmx5OyBodW1hbiByZXZpZXcgc3RpbGwgcmVxdWlyZWQuIE5vIHRyYWluaW5nIHN0YXJ0ZWQuJykKICAgIHJldHVybiBkZXN0CgpkZWYgcHVibGlzaChmb2xkZXIsIHRva2VuLCBwcmVmaXgpOgogICAgIiIiT25lIGNvbW1pdCBwZXIgbWFqb3IgYWN0aW9uOyBzbWFsbCBpbW11dGFibGUgY29udGVudC4gTm8gY2xhaW0vaGVhcnRiZWF0IHdyaXRlcy4iIiIKICAgIGZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBIZkFwaQogICAgZm9sZGVyPVBhdGgoZm9sZGVyKQogICAgYWxsb3dlZD1bJ1NUQVRVUy5qc29uJywnTUFOSUZFU1QuanNvbicsJ1BJTE9UXzEyX0lNQUdFUy56aXAnLCdSRVZJRVcuanNvbicsJ0FOTk9UQVRJT05TLmpzb24nLAogICAgICAgICAgICAgJ3M5X3BpbG90LnB5JywncGlsb3RfdGVtcGxhdGUuaHRtbCddCiAgICBmaWxlcz1bcCBmb3IgcCBpbiBmb2xkZXIuaXRlcmRpcigpIGlmIHAubmFtZSBpbiBhbGxvd2VkIGFuZCBwLmlzX2ZpbGUoKV0KICAgIGlmIHN1bShwLnN0YXQoKS5zdF9zaXplIGZvciBwIGluIGZpbGVzKT5NQVhfQllURVM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVXBsb2FkIGV4Y2VlZHMgMjAgTWlCIGxpbWl0JykKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDUpOgogICAgICAgIHRyeToKICAgICAgICAgICAgcmVzdWx0PUhmQXBpKHRva2VuPXRva2VuKS51cGxvYWRfZm9sZGVyKHJlcG9faWQ9UkVQTyxyZXBvX3R5cGU9J2RhdGFzZXQnLGZvbGRlcl9wYXRoPXN0cihmb2xkZXIpLAogICAgICAgICAgICAgICAgcGF0aF9pbl9yZXBvPXByZWZpeCxhbGxvd19wYXR0ZXJucz1hbGxvd2VkLGNvbW1pdF9tZXNzYWdlPSdTOSBzbWFsbCBhbm5vdGF0aW9uIHBpbG90OyBubyBtb2RlbCB0cmFpbmluZycpCiAgICAgICAgICAgIHByaW50KCdIRiBwdWJsaWNhdGlvbiBzdWNjZWVkZWQ6JyxyZXN1bHQub2lkKTsgcmV0dXJuIHJlc3VsdC5vaWQKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgcmVzcG9uc2U9Z2V0YXR0cihleGMsJ3Jlc3BvbnNlJyxOb25lKTsgc3RhdHVzPWdldGF0dHIocmVzcG9uc2UsJ3N0YXR1c19jb2RlJyxOb25lKQogICAgICAgICAgICBpZiBzdGF0dXMgbm90IGluICg0MjksNTAwLDUwMiw1MDMsNTA0KSBvciBhdHRlbXB0PT00OgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdIRiBwdWJsaWNhdGlvbiBkaWQgbm90IGNvbXBsZXRlLiBLZWVwIGxvY2FsIG91dHB1dHMgYW5kIHJldHJ5IHRoaXMgY2VsbDsgbm8gdHJhaW5pbmcgd2FzIHJ1bi4nKSBmcm9tIE5vbmUKICAgICAgICAgICAgaGludD1nZXRhdHRyKHJlc3BvbnNlLCdoZWFkZXJzJyx7fSkuZ2V0KCdSZXRyeS1BZnRlcicsJycpCiAgICAgICAgICAgIHdhaXQ9bWF4KDEwKjIqKmF0dGVtcHQsZmxvYXQoaGludCkgaWYgc3RyKGhpbnQpLmlzZGlnaXQoKSBlbHNlIDApCiAgICAgICAgICAgIHByaW50KGYnSEYgYmFja29mZjoge3dhaXQ6LjBmfXMuIFNhdmVkIGxvY2FsIG91dHB1dHMgcmVtYWluIGF2YWlsYWJsZS4nKQogICAgICAgICAgICB1bnRpbD10aW1lLm1vbm90b25pYygpK3dhaXQKICAgICAgICAgICAgd2hpbGUgdGltZS5tb25vdG9uaWMoKTx1bnRpbDogdGltZS5zbGVlcChtaW4oNSxtYXgoMCx1bnRpbC10aW1lLm1vbm90b25pYygpKSkpCg=='))
(WORK/'s9_geometry.py').write_bytes(base64.b64decode('IiIiQ1BVLW9ubHksIHByb3ZlbmFuY2UtcGlubmVkIHBpbG90IGNvbXBhcmlzb24gdXNpbmcgYWxyZWFkeSBwdWJsaXNoZWQgcHJlZGljdGlvbnMuIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHJlcXVlc3RzCmZyb20gUElMIGltcG9ydCBJbWFnZSwgSW1hZ2VEcmF3CmltcG9ydCBzOV9waWxvdCBhcyBwCgpWRVJTSU9OID0gJ3M5LWdlb21ldHJ5LXBpbG90LXIxJwpTT1VSQ0UgPSAnMDViZjM3YzBmMjA2N2IwMTE5ZWYwNTdhMjQ0MmQ3NzU5Y2Y5YWM1MScKUEFDS0FHRSA9ICc4NzkzM2NhY2M5Y2MwNjIzNGRjMTUwZmJiYTc0OWU0NjlkODFlNTc4NTA2NjZkNTFhMmY0NTVhNzQ2ZDNmZDk2JwpBTk5PVEFUSU9OID0gJzQxMTBiZGNjNDEzZmYzMjdkN2FiMmU2ZDEwN2VmYWMwNzEwMDhkYzczYjFiZjA0MTQzYTZjOWExMTY1MTZlMTgnClBMQU4gPSAnMWY2Njk0NTc3MjUzZTAwNTRmN2EyMmRmNmVjNTJkMzA3OTcwNjNjZjQ5OGIzNDViZDcxZmU5Yjk4YmFiOTNkZicKUzUgPSBmJ3M1L3M1LW1hbnVhbC0yMDI2LTA5LTEwLXIxL3tQTEFOfScKCmRlZiBmZXRjaChwYXRoLCBjYWNoZSk6CiAgICAiIiJQdWJsaWMgaW1tdXRhYmxlIHNvdXJjZXMgb25seTsgYm91bmRlZCBkb3dubG9hZHM7IGF0b21pYyBjYWNoZS4iIiIKICAgIGNhY2hlPVBhdGgoY2FjaGUpOyBjYWNoZS5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIGRlc3Q9Y2FjaGUvKHAuZGlnZXN0KHBhdGguZW5jb2RlKCkpKycuanNvbicpCiAgICBpZiBkZXN0LmV4aXN0cygpOiByZXR1cm4gZGVzdC5yZWFkX2J5dGVzKCkKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDUpOgogICAgICAgIHRyeToKICAgICAgICAgICAgd2l0aCByZXF1ZXN0cy5nZXQoZidodHRwczovL2h1Z2dpbmdmYWNlLmNvL2RhdGFzZXRzL3twLlJFUE99L3Jlc29sdmUve1NPVVJDRX0ve3BhdGh9JywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdGltZW91dD00NSxzdHJlYW09VHJ1ZSkgYXMgcmVzcG9uc2U6CiAgICAgICAgICAgICAgICByZXNwb25zZS5yYWlzZV9mb3Jfc3RhdHVzKCk7IHJhdz1ieXRlYXJyYXkoKQogICAgICAgICAgICAgICAgZm9yIGNodW5rIGluIHJlc3BvbnNlLml0ZXJfY29udGVudCg2NTUzNik6CiAgICAgICAgICAgICAgICAgICAgcmF3LmV4dGVuZChjaHVuaykKICAgICAgICAgICAgICAgICAgICBpZiBsZW4ocmF3KT4yKjEwMjQqKjI6IHJhaXNlIFZhbHVlRXJyb3IoJ1NvdXJjZSBleGNlZWRzIDIgTWlCIHBlci1maWxlIGNhcCcpCiAgICAgICAgICAgIGlmIHN1bSh4LnN0YXQoKS5zdF9zaXplIGZvciB4IGluIGNhY2hlLmdsb2IoJyouanNvbicpKStsZW4ocmF3KT4yMCoxMDI0KioyOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignU291cmNlIGNhY2hlIGV4Y2VlZHMgMjAgTWlCIGJ1ZGdldCcpCiAgICAgICAgICAgIGpzb24ubG9hZHMocmF3KQogICAgICAgICAgICB0bXA9ZGVzdC53aXRoX3N1ZmZpeCgnLnRtcCcpOyB0bXAud3JpdGVfYnl0ZXMocmF3KTsgdG1wLnJlcGxhY2UoZGVzdCkKICAgICAgICAgICAgcmV0dXJuIGJ5dGVzKHJhdykKICAgICAgICBleGNlcHQgcmVxdWVzdHMuUmVxdWVzdEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgIHN0YXR1cz1nZXRhdHRyKGV4Yy5yZXNwb25zZSwnc3RhdHVzX2NvZGUnLE5vbmUpCiAgICAgICAgICAgIGlmIHN0YXR1cyBpcyBub3QgTm9uZSBhbmQgc3RhdHVzIG5vdCBpbiAoNDI5LDUwMCw1MDIsNTAzLDUwNCk6IHJhaXNlCiAgICAgICAgICAgIGlmIGF0dGVtcHQ9PTQ6IHJhaXNlCiAgICAgICAgICAgIGhpbnQ9Z2V0YXR0cihleGMucmVzcG9uc2UsJ2hlYWRlcnMnLHt9KS5nZXQoJ1JldHJ5LUFmdGVyJywnMCcpCiAgICAgICAgICAgIHRpbWUuc2xlZXAobWF4KDUqMioqYXR0ZW1wdCxmbG9hdChoaW50KSBpZiBoaW50LmlzZGlnaXQoKSBlbHNlIDApKQoKZGVmIGJvdW5kYXJ5KG1hc2ssIHNpZGUsIHkpOgogICAgeHM9bnAuZmxhdG5vbnplcm8obWFza1t5XSkKICAgIGlmIG5vdCBsZW4oeHMpOiByZXR1cm4gTm9uZSwnZW1wdHlfcm93JwogICAgeD1pbnQoeHNbMF0gaWYgc2lkZT09J2xlZnQnIGVsc2UgeHNbLTFdKQogICAgaWYgeCBpbiAoMCxtYXNrLnNoYXBlWzFdLTEpOiByZXR1cm4gTm9uZSwnZnJhbWVfY2xpcHBlZCcKICAgIHJldHVybiB4LCd2aXNpYmxlJwoKZGVmIHJvd3NfZm9yKHJlZiwgYW5ub3RhdGlvbiwgbWFzaywgY29tcGxldGUpOgogICAgcm93cz1bXQogICAgZm9yIGksbmFtZSBpbiBlbnVtZXJhdGUocC5QT0lOVFMpOgogICAgICAgIHBvaW50PWFubm90YXRpb24uZ2V0KCdwb2ludHMnLHt9KS5nZXQobmFtZSx7fSkKICAgICAgICB4LHN0YXRlPWJvdW5kYXJ5KG1hc2ssbmFtZS5zcGxpdCgnXycpWzBdLHJlZlsnZ3VpZGVfeSddW2kvLzJdKQogICAgICAgIHJlYXNvbj0oJ1AwNl9yZXZpZXdfaG9sZCcgaWYgcmVmWydwaWxvdF9pZCddPT0nUDA2JyBlbHNlCiAgICAgICAgICAgICAgICAnaW5jb21wbGV0ZV9hbm5vdGF0aW9uJyBpZiBub3QgY29tcGxldGUgZWxzZQogICAgICAgICAgICAgICAgJ2xhYmVsX25vdF92aXNpYmxlJyBpZiBwb2ludC5nZXQoJ3N0YXRlJykhPSd2aXNpYmxlJyBlbHNlICcnKQogICAgICAgIGVsaWdpYmxlPW5vdCByZWFzb24KICAgICAgICBzY29yZWQ9ZWxpZ2libGUgYW5kIHN0YXRlPT0ndmlzaWJsZScKICAgICAgICByb3dzLmFwcGVuZChkaWN0KHBpbG90X2lkPXJlZlsncGlsb3RfaWQnXSxwb2ludD1uYW1lLHk9cmVmWydndWlkZV95J11baS8vMl0sCiAgICAgICAgICAgIGxhYmVsX3N0YXRlPXBvaW50LmdldCgnc3RhdGUnLCdtaXNzaW5nJyksbGFiZWxfeD1wb2ludC5nZXQoJ3gnKSxwcmVkaWN0ZWRfeD14LAogICAgICAgICAgICBwcmVkaWN0aW9uX3N0YXRlPXN0YXRlLGVsaWdpYmxlPWVsaWdpYmxlLHNjb3JlZD1zY29yZWQsZXhjbHVzaW9uPXJlYXNvbiwKICAgICAgICAgICAgZXJyb3JfcHg9YWJzKHgtcG9pbnRbJ3gnXSkgaWYgc2NvcmVkIGVsc2UgTm9uZSwKICAgICAgICAgICAgZXJyb3Jfd2lkdGhfZnJhY3Rpb249YWJzKHgtcG9pbnRbJ3gnXSkvcmVmWyd3aWR0aCddIGlmIHNjb3JlZCBlbHNlIE5vbmUpKQogICAgcmV0dXJuIHJvd3MKCmRlZiBzdW1tYXJ5KHJvd3MpOgogICAgZWxpZ2libGU9W3IgZm9yIHIgaW4gcm93cyBpZiByWydlbGlnaWJsZSddXTsgc2NvcmVkPVtyIGZvciByIGluIHJvd3MgaWYgclsnc2NvcmVkJ11dCiAgICByZXR1cm4gZGljdCh0b3RhbF9wb2ludHM9bGVuKHJvd3MpLGVsaWdpYmxlX3Zpc2libGVfcG9pbnRzPWxlbihlbGlnaWJsZSksc2NvcmVkX3BvaW50cz1sZW4oc2NvcmVkKSwKICAgICAgICBjb3ZlcmFnZT1sZW4oc2NvcmVkKS9sZW4oZWxpZ2libGUpIGlmIGVsaWdpYmxlIGVsc2UgTm9uZSwKICAgICAgICBtZWRpYW5fZXJyb3JfcHg9ZmxvYXQobnAubWVkaWFuKFtyWydlcnJvcl9weCddIGZvciByIGluIHNjb3JlZF0pKSBpZiBzY29yZWQgZWxzZSBOb25lLAogICAgICAgIG1lYW5fZXJyb3Jfd2lkdGhfZnJhY3Rpb249ZmxvYXQobnAubWVhbihbclsnZXJyb3Jfd2lkdGhfZnJhY3Rpb24nXSBmb3IgciBpbiBzY29yZWRdKSkgaWYgc2NvcmVkIGVsc2UgTm9uZSwKICAgICAgICByZWplY3RlZF9lbGlnaWJsZV9wb2ludHM9bGVuKGVsaWdpYmxlKS1sZW4oc2NvcmVkKSkKCmRlZiBwdWJsaXNoKG91dCwgdG9rZW4pOgogICAgZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IEhmQXBpCiAgICBvdXQ9UGF0aChvdXQpCiAgICBpZiBzdW0oZi5zdGF0KCkuc3Rfc2l6ZSBmb3IgZiBpbiBvdXQucmdsb2IoJyonKSBpZiBmLmlzX2ZpbGUoKSk+MjAqMTAyNCoqMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdPdXRwdXQgdXBsb2FkIGV4Y2VlZHMgMjAgTWlCJykKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDUpOgogICAgICAgIHRyeToKICAgICAgICAgICAgcmVzdWx0PUhmQXBpKHRva2VuPXRva2VuKS51cGxvYWRfZm9sZGVyKHJlcG9faWQ9cC5SRVBPLHJlcG9fdHlwZT0nZGF0YXNldCcsCiAgICAgICAgICAgICAgICBmb2xkZXJfcGF0aD1zdHIob3V0KSxwYXRoX2luX3JlcG89ZidzOS97VkVSU0lPTn0ve291dC5uYW1lfScsCiAgICAgICAgICAgICAgICBjb21taXRfbWVzc2FnZT0nUzkgc2F2ZWQtbWFzayBnZW9tZXRyeSBwaWxvdDsgbm8gdHJhaW5pbmcnKQogICAgICAgICAgICBwcmludCgnSEYgcHVibGljYXRpb24gc3VjY2VlZGVkOicscmVzdWx0Lm9pZCxmbHVzaD1UcnVlKTsgcmV0dXJuIHJlc3VsdC5vaWQKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgc3RhdHVzPWdldGF0dHIoZ2V0YXR0cihleGMsJ3Jlc3BvbnNlJyxOb25lKSwnc3RhdHVzX2NvZGUnLE5vbmUpCiAgICAgICAgICAgIGlmIHN0YXR1cyBub3QgaW4gKE5vbmUsNDI5LDUwMCw1MDIsNTAzLDUwNCkgb3IgYXR0ZW1wdD09NDogcmFpc2UKICAgICAgICAgICAgaGludD1nZXRhdHRyKGdldGF0dHIoZXhjLCdyZXNwb25zZScsTm9uZSksJ2hlYWRlcnMnLHt9KS5nZXQoJ1JldHJ5LUFmdGVyJywnMCcpCiAgICAgICAgICAgIHdhaXQ9bWF4KDEwKjIqKmF0dGVtcHQsZmxvYXQoaGludCkgaWYgaGludC5pc2RpZ2l0KCkgZWxzZSAwKQogICAgICAgICAgICBwcmludChmJ1VwbG9hZCByZXRyeSBpbiB7d2FpdH1zOyBsb2NhbCByZXN1bHRzIGFyZSBzYWZlLicsZmx1c2g9VHJ1ZSk7IHRpbWUuc2xlZXAod2FpdCkKCmRlZiBydW4od29yaywgdG9rZW49Tm9uZSwgdXBsb2FkPVRydWUpOgogICAgZnJvbSBweWNvY290b29scyBpbXBvcnQgbWFzayBhcyBjb2NvCiAgICB3b3JrPVBhdGgod29yayk7IGNhY2hlPXdvcmsvJ3NvdXJjZV9jYWNoZScKICAgIHByb3ZlbmFuY2U9e30KICAgIGRlZiByZWFkKHBhdGgpOgogICAgICAgIHJhdz1mZXRjaChwYXRoLGNhY2hlKTsgcHJvdmVuYW5jZVtwYXRoXT1wLmRpZ2VzdChyYXcpOyByZXR1cm4ganNvbi5sb2FkcyhyYXcpCiAgICBiYXNlPWYnczkve3AuVkVSU0lPTn0ve1BBQ0tBR0V9JwogICAgbWFuaWZlc3Q9cmVhZChiYXNlKycvTUFOSUZFU1QuanNvbicpCiAgICBhc3NlcnQgcC5kaWdlc3QocC5jYW5vbmljYWwoe2s6diBmb3Igayx2IGluIG1hbmlmZXN0Lml0ZW1zKCkgaWYgayE9J3BhY2thZ2VfaWQnfSkpPT1QQUNLQUdFCiAgICBhcGF0aD1mJ3tiYXNlfS9yZXZpZXdzL3tBTk5PVEFUSU9OfS9BTk5PVEFUSU9OUy5qc29uJwogICAgYW5ub3RhdGlvbj1yZWFkKGFwYXRoKTsgYXNzZXJ0IHByb3ZlbmFuY2VbYXBhdGhdPT1BTk5PVEFUSU9OCiAgICB2YWxpZGF0ZWQ9cC52YWxpZGF0ZV9hbm5vdGF0aW9ucyhhbm5vdGF0aW9uLG1hbmlmZXN0KQogICAgbGFiZWxzPXthWydwaWxvdF9pZCddOmEgZm9yIGEgaW4gYW5ub3RhdGlvblsnYW5ub3RhdGlvbnMnXX0KICAgIGNoZWNrcz17YVsncGlsb3RfaWQnXTphIGZvciBhIGluIHZhbGlkYXRlZH0KICAgIHBsYW49cmVhZChTNSsnL3Byb3RvY29sLmpzb24nKTsgYXNzZXJ0IHAuZGlnZXN0KHAuY2Fub25pY2FsKHBsYW4pKT09UExBTgogICAgY29udHJhY3Q9ZGljdCh2ZXJzaW9uPVZFUlNJT04sc291cmNlX3JldmlzaW9uPVNPVVJDRSxhbm5vdGF0aW9uX3NoYTI1Nj1BTk5PVEFUSU9OLAogICAgICAgIHBsYW5faGFzaD1QTEFOLG1vZGVsPSdzZWdmb3JtZXJfYjAnLHNlZWQ9MSxlbmRwb2ludD0nc2F2ZWRfZXBvY2g2MF9wcmVkaWN0aW9ucycsCiAgICAgICAgc291cmNlX3NoYTI1Nj1wLmRpZ2VzdChQYXRoKF9fZmlsZV9fKS5yZWFkX2J5dGVzKCkpLAogICAgICAgIHBpbG90X3ZhbGlkYXRvcl9zaGEyNTY9cC5kaWdlc3QoUGF0aChwLl9fZmlsZV9fKS5yZWFkX2J5dGVzKCkpLAogICAgICAgIHJldmlld19ob2xkPVsnUDA2J10sZWRnZV9ydWxlPSdyZWplY3QgZXhhY3QgbmF0aXZlIGZyYW1lIGVkZ2UnLAogICAgICAgIGxpbWl0YXRpb25zPVsncGlsb3QgZmVhc2liaWxpdHkgb25seTsgbm8gaW5kZXBlbmRlbnQgbmV3LXR5cmUgYWNjdXJhY3knLAogICAgICAgICAgJ2tub3duIGZvbGQgMC8yIGlkZW50aXR5IGxlYWthZ2UgY2F2ZWF0IHJldGFpbmVkJywKICAgICAgICAgICdtYXNrIGJvdW5kYXJ5IGFuZCBwcm9wb3NlZCB0cmVhZCB0cmFuc2l0aW9uIG1heSBkaWZmZXInLAogICAgICAgICAgJ25vIHBoeXNpY2FsIGFuZ2xlcywgaGVhbHRoeSBjZXJ0aWZpY2F0aW9uIG9yIEhSTmV0IGNvbXBhcmlzb24nXSkKICAgIG91dD13b3JrLydyZXN1bHRzJy9wLmRpZ2VzdChwLmNhbm9uaWNhbChjb250cmFjdCkpOyBvdXQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBwLndyaXRlX2pzb24ob3V0LydDT05UUkFDVC5qc29uJyxjb250cmFjdCkKICAgIHAud3JpdGVfanNvbihvdXQvJ0FOTk9UQVRJT05TLmpzb24nLGFubm90YXRpb24pCiAgICBwLndyaXRlX2pzb24ob3V0LydNQU5JRkVTVC5qc29uJyxtYW5pZmVzdCkKICAgIChvdXQvJ3M5X2dlb21ldHJ5LnB5Jykud3JpdGVfYnl0ZXMoUGF0aChfX2ZpbGVfXykucmVhZF9ieXRlcygpKQogICAgKG91dC8nczlfcGlsb3QucHknKS53cml0ZV9ieXRlcyhQYXRoKHAuX19maWxlX18pLnJlYWRfYnl0ZXMoKSkKICAgIHJvd3M9W107IGxhc3RfcHVzaD10aW1lLm1vbm90b25pYygpCiAgICB0cnk6CiAgICAgICAgZm9yIHJlZiBpbiBtYW5pZmVzdFsnaW1hZ2VzJ106CiAgICAgICAgICAgIGZvbGQ9cmVmWydmb2xkJ107IHNwbGl0PXBsYW5bJ2RhdGEnXVsnc3BsaXRzJ11bc3RyKGZvbGQpXTsgaWlkPXJlZlsnaW1hZ2VfaWQnXQogICAgICAgICAgICBhc3NlcnQgaWlkIGluIHNwbGl0Wyd2YWxpZGF0aW9uJ10gYW5kIGlpZCBub3QgaW4gc3BsaXRbJ3RyYWluJ10sICdOb3QgaGVsZCBvdXQgZnJvbSB0aGlzIHJ1bicKICAgICAgICAgICAgc291cmNlX3Jvdz1uZXh0KHIgZm9yIHIgaW4gcGxhblsnZGF0YSddWydyZWNvcmRzJ10gaWYgclsnaW1hZ2VfaWQnXT09aWlkKQogICAgICAgICAgICBhc3NlcnQgc291cmNlX3Jvd1snaW1hZ2Vfc2hhMjU2J109PXJlZlsnaW1hZ2Vfc2hhMjU2J10KICAgICAgICAgICAgam9iPWRpY3QobW9kZWw9J3NlZ2Zvcm1lcl9iMCcsYmFja2VuZD0nc2VtYW50aWMnLGZvbGQ9Zm9sZCxzZWVkPTEsCiAgICAgICAgICAgICAgICAgICAgIHJ1bl9pZD1mJ3NlZ2Zvcm1lcl9iMC1me2ZvbGR9LXMxJykKICAgICAgICAgICAgcHJlZml4PVM1KycvcnVucy8nK2pvYlsncnVuX2lkJ10KICAgICAgICAgICAgc3RhdHVzPXJlYWQocHJlZml4KycvU1RBVFVTLmpzb24nKQogICAgICAgICAgICBhc3NlcnQgc3RhdHVzLmdldCgnZXBvY2gnKT09NjAgYW5kIHN0YXR1cy5nZXQoJ2V2YWx1YXRlZCcpIGlzIFRydWUsICdFeHBlY3RlZCBldmFsdWF0ZWQgZmluYWwgZXBvY2ggNjAnCiAgICAgICAgICAgIGFzc2VydCBzdGF0dXNbJ2pvYiddPT1qb2IgYW5kIHN0YXR1c1sncGxhbl9oYXNoJ109PVBMQU4KICAgICAgICAgICAgcmVjPXJlYWQocHJlZml4KycvcHJlZGljdGlvbnMvJytpaWQrJy5qc29uJykKICAgICAgICAgICAgYXNzZXJ0IHJlY1snam9iJ109PWpvYiBhbmQgcmVjWydwbGFuX2hhc2gnXT09UExBTiBhbmQgcmVjWydpbWFnZV9pZCddPT1paWQKICAgICAgICAgICAgYXNzZXJ0IChyZWNbJ2hlaWdodCddLHJlY1snd2lkdGgnXSk9PShyZWZbJ2hlaWdodCddLHJlZlsnd2lkdGgnXSkKICAgICAgICAgICAgbWFza3M9W2RldFsnbWFzayddIGZvciBkZXQgaW4gcmVjWydkZXRlY3Rpb25zJ10gaWYgZGV0WydsYWJlbCddPT0xXQogICAgICAgICAgICBhc3NlcnQgbGVuKG1hc2tzKTw9MSwgJ1NlbWFudGljIG91dHB1dCBzaG91bGQgY29udGFpbiBhdCBtb3N0IG9uZSB0cmVhZCBtYXNrJwogICAgICAgICAgICBtYXNrPW5wLnplcm9zKChyZWZbJ2hlaWdodCddLHJlZlsnd2lkdGgnXSksZHR5cGU9Ym9vbCkKICAgICAgICAgICAgaWYgbWFza3M6CiAgICAgICAgICAgICAgICBhc3NlcnQgbWFza3NbMF1bJ3NpemUnXT09W3JlZlsnaGVpZ2h0J10scmVmWyd3aWR0aCddXQogICAgICAgICAgICAgICAgbWFzaz1jb2NvLmRlY29kZShtYXNrc1swXSkuYXN0eXBlKGJvb2wpCiAgICAgICAgICAgIHBhcnQ9cm93c19mb3IocmVmLGxhYmVscy5nZXQocmVmWydwaWxvdF9pZCddLHt9KSxtYXNrLGNoZWNrcy5nZXQocmVmWydwaWxvdF9pZCddLHt9KS5nZXQoJ2NvbXBsZXRlJyxGYWxzZSkpCiAgICAgICAgICAgIGZvciByb3cgaW4gcGFydDogcm93LnVwZGF0ZShydW5faWQ9am9iWydydW5faWQnXSxmb2xkPWZvbGQsaW1hZ2VfaWQ9aWlkKQogICAgICAgICAgICByb3dzLmV4dGVuZChwYXJ0KQogICAgICAgICAgICBwaWN0dXJlPUltYWdlLmZyb21hcnJheSgobWFzay5hc3R5cGUoJ3VpbnQ4JykqMTUwKSkuY29udmVydCgnUkdCJyk7IGRyYXc9SW1hZ2VEcmF3LkRyYXcocGljdHVyZSkKICAgICAgICAgICAgZm9yIHJvdyBpbiBwYXJ0OgogICAgICAgICAgICAgICAgeT1yb3dbJ3knXTsgZHJhdy5saW5lKCgwLHkscmVmWyd3aWR0aCddLTEseSksZmlsbD0nZ3JheScpCiAgICAgICAgICAgICAgICBmb3IgeCxjb2xvciBpbiBbKHJvd1snbGFiZWxfeCddLCdjeWFuJyksKHJvd1sncHJlZGljdGVkX3gnXSwneWVsbG93JyldOgogICAgICAgICAgICAgICAgICAgIGlmIHggaXMgbm90IE5vbmU6IGRyYXcuZWxsaXBzZSgoeC03LHktNyx4KzcseSs3KSxvdXRsaW5lPWNvbG9yLHdpZHRoPTMpCiAgICAgICAgICAgIHBpY3R1cmUudGh1bWJuYWlsKCg1NzYsNzY4KSk7IHBpY3R1cmUuc2F2ZShvdXQvKHJlZlsncGlsb3RfaWQnXSsnLnBuZycpKQogICAgICAgICAgICBwLndyaXRlX2pzb24ob3V0LydQT0lOVFMuanNvbicscm93cykKICAgICAgICAgICAgcC53cml0ZV9qc29uKG91dC8nUFJPVkVOQU5DRS5qc29uJyxwcm92ZW5hbmNlKQogICAgICAgICAgICBwLndyaXRlX2pzb24ob3V0LydTVEFUVVMuanNvbicsZGljdChzdGF0dXM9J3BhcnRpYWwnLGltYWdlc19wcm9jZXNzZWQ9bGVuKHJvd3MpLy82LHRyYWluaW5nPUZhbHNlKSkKICAgICAgICAgICAgcHJpbnQoZiJ7cmVmWydwaWxvdF9pZCddfSB8IHtqb2JbJ3J1bl9pZCddfSB8IHtzdW1tYXJ5KHBhcnQpfSIsZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgaWYgdXBsb2FkIGFuZCB0aW1lLm1vbm90b25pYygpLWxhc3RfcHVzaD49MTgwMDoKICAgICAgICAgICAgICAgIHB1Ymxpc2gob3V0LHRva2VuKTsgbGFzdF9wdXNoPXRpbWUubW9ub3RvbmljKCkKICAgICAgICByZXN1bHQ9c3VtbWFyeShyb3dzKQogICAgICAgIHJlc3VsdC51cGRhdGUocGVyX2ltYWdlPXtyWydwaWxvdF9pZCddOnN1bW1hcnkoW3ggZm9yIHggaW4gcm93cyBpZiB4WydwaWxvdF9pZCddPT1yWydwaWxvdF9pZCddXSkgZm9yIHIgaW4gbWFuaWZlc3RbJ2ltYWdlcyddfSwKICAgICAgICAgICAgZGVjaXNpb249J1JldmlldyBjb3ZlcmFnZSBhbmQgb3ZlcmxheXM7IG5vIGF1dG9tYXRpYyBIUk5ldCBhcHByb3ZhbCBvciByZWplY3Rpb24uJywKICAgICAgICAgICAgZnVsbF9zOV9jb21wbGV0ZT1GYWxzZSxoZWFsdGh5X3JlZmVyZW5jZV9lbGlnaWJsZT1GYWxzZSkKICAgICAgICBwLndyaXRlX2pzb24ob3V0LydTVU1NQVJZLmpzb24nLHJlc3VsdCkKICAgICAgICBwLndyaXRlX2pzb24ob3V0LydTVEFUVVMuanNvbicsZGljdChzdGF0dXM9J2NvbXBsZXRlJyxpbWFnZXNfcHJvY2Vzc2VkPTEyLHRyYWluaW5nPUZhbHNlLGZ1bGxfczlfY29tcGxldGU9RmFsc2UpKQogICAgICAgIChvdXQvJ1JFQURNRS5tZCcpLndyaXRlX3RleHQoJyMgU2F2ZWQtbWFzayBnZW9tZXRyeSBwaWxvdFxuXG5DUFUgcmV1c2Ugb2YgcHVibGlzaGVkIFNlZ0Zvcm1lci1CMCBzZWVkLTEgbmF0aXZlIHByZWRpY3Rpb25zLiBObyBuZXcgaW5mZXJlbmNlIG9yIHRyYWluaW5nLlxuXG5QTkc6IGdyZXkgaXMgcHJlZGljdGVkIHRyZWFkIG1hc2ssIGN5YW4gaHVtYW4gY2xpY2tzLCB5ZWxsb3cgYWNjZXB0ZWQgcHJlZGljdGVkIGJvdW5kYXJ5LiBQMDYgaXMgZGlhZ25vc3RpYyBvbmx5IGFuZCBleGNsdWRlZC4gRW1wdHkvY2xpcHBlZCBwcmVkaWN0aW9ucyByZWR1Y2UgY292ZXJhZ2U7IHRoZXkgYXJlIG5vdCB6ZXJvLWVycm9yIHN1Y2Nlc3Nlcy5cblxuUmVhZCBTVU1NQVJZLmpzb24sIFBPSU5UUy5qc29uIGFuZCBDT05UUkFDVC5qc29uIHRvZ2V0aGVyLiBQb2ludCBlcnJvcnMgYXJlIGNvbmRpdGlvbmFsIG9uIGFjY2VwdGVkIHByZWRpY3Rpb25zOyBkbyBub3QgaW50ZXJwcmV0IHRoZSBtZWFuIHdpdGhvdXQgY292ZXJhZ2UuIE5vIGF1dG9tYXRpYyBIUk5ldCBkZWNpc2lvbi5cbicsZW5jb2Rpbmc9J3V0Zi04JykKICAgICAgICBpZiB1cGxvYWQ6IHB1Ymxpc2gob3V0LHRva2VuKQogICAgICAgIHByaW50KGpzb24uZHVtcHMocmVzdWx0LGluZGVudD0yKSk7IHJldHVybiBvdXQKICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uOgogICAgICAgIGlmIHVwbG9hZDoKICAgICAgICAgICAgdHJ5OiBwdWJsaXNoKG91dCx0b2tlbikKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6IHByaW50KCdTdG9wL2Vycm9yIGZsdXNoIGZhaWxlZDsgcmV0YWluIGxvY2FsIG91dHB1dCBhbmQgcmVydW46Jyx0eXBlKGV4YykuX19uYW1lX18pCiAgICAgICAgcmFpc2UK'))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.7/411.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.8/248.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 5.0.0 requires fsspec[http]<=2026.4.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2

10572

In [2]:
import os, signal, time
from kaggle_secrets import UserSecretsClient
token=UserSecretsClient().get_secret('HF_TOKEN')
if not token: raise RuntimeError('Enable HF_TOKEN in Kaggle Secrets.')
env=os.environ.copy()
env['HF_TOKEN']=token
env['PYTHONPATH']=str(DEPS)+os.pathsep+str(WORK)
code="import os,s9_geometry as g; g.run('/kaggle/working/s9_geometry',token=os.environ['HF_TOKEN'])"
child=subprocess.Popen([sys.executable,'-u','-c',code],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
del token, env
try:
    for line in child.stdout:
        print(line,end='',flush=True)
    result=child.wait()
except KeyboardInterrupt:
    if child.poll() is None:
        child.send_signal(signal.SIGINT)
        print('Stop requested: allowing child to flush completed results to HF.')
        for line in child.stdout:
            print(line,end='',flush=True)
        child.wait()
    raise
if result: raise RuntimeError('Comparison/upload did not finish. Read the preceding error; local results are retained. Retry Run All.')
from IPython.display import FileLink, display
for folder in sorted((WORK/'results').iterdir()):
    display(FileLink(str(folder/'SUMMARY.json')))
    display(FileLink(str(folder/'POINTS.json')))
print('Done. Send this executed notebook for HF verification. No HRNet training started.')


P01 | segformer_b0-f2-s1 | {'total_points': 6, 'eligible_visible_points': 6, 'scored_points': 6, 'coverage': 1.0, 'median_error_px': 6.0, 'mean_error_width_fraction': 0.004195601851851852, 'rejected_eligible_points': 0}
P02 | segformer_b0-f1-s1 | {'total_points': 6, 'eligible_visible_points': 6, 'scored_points': 6, 'coverage': 1.0, 'median_error_px': 17.0, 'mean_error_width_fraction': 0.015914351851851853, 'rejected_eligible_points': 0}
P03 | segformer_b0-f1-s1 | {'total_points': 6, 'eligible_visible_points': 6, 'scored_points': 6, 'coverage': 1.0, 'median_error_px': 11.5, 'mean_error_width_fraction': 0.00882523148148148, 'rejected_eligible_points': 0}
P04 | segformer_b0-f2-s1 | {'total_points': 6, 'eligible_visible_points': 6, 'scored_points': 6, 'coverage': 1.0, 'median_error_px': 20.5, 'mean_error_width_fraction': 0.016493055555555556, 'rejected_eligible_points': 0}
P05 | segformer_b0-f0-s1 | {'total_points': 6, 'eligible_visible_points': 6, 'scored_points': 6, 'coverage': 1.0, 'med

/kaggle/working/s9_geometry/results/929ae395f3521f988680e8c4019559e34241f2b1cd9f998ff48605a3afebc20b/SUMMARY.json

/kaggle/working/s9_geometry/results/929ae395f3521f988680e8c4019559e34241f2b1cd9f998ff48605a3afebc20b/POINTS.json

Done. Send this executed notebook for HF verification. No HRNet training started.
